### Creating Simple GEN AI App with Langchain

- We have a specific website where we have some content. we are going to extract the data from that website

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['GROQ_API_KEY']=os.getenv("GROQ_API_KEY")

## langsmith Tracking
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")

In [29]:
## Data Ingestion - From Website we need to scrape the data for that we need beautifulsoup4 in our environment

from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://docs.langchain.com/langsmith/manage-prompts-programmatically")
print(loader)
print("-----------------------------------------------------------------")
docs = loader.load()
docs

-----------------------------------------------------------------


[Document(metadata={'source': 'https://docs.langchain.com/langsmith/manage-prompts-programmatically', 'title': 'Manage prompts programmatically - Docs by LangChain', 'language': 'en'}, page_content='Manage prompts programmatically - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationCreate and update promptsManage prompts programmaticallyGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewQuickstartConceptsPolly AI assistantBetaModel providersCreate and update promptsCreate a promptManage promptsManage prompts programmaticallyPrompt template formatConfigure prompt settingsUse tools in a promptInclude multimodal content in a promptWrite your prompt with AIConnect to modelsTutorialsOptimize a classifierSync prompts with GitHubTest multi-turn conversationsOn this pageInstall packagesConfigure environment variablesPush a promptPull a promptPrompt cachingD

In [30]:
# 3. Access the content
if docs:
    # Each document object has a page_content attribute with the text
    print(docs[0].page_content[:500]) # Print the first 500 characters
    print(f"\nSource: {docs[0].metadata['source']}")
else:
    print("Failed to load documents.")


Manage prompts programmatically - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationCreate and update promptsManage prompts programmaticallyGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewQuickstartConceptsPolly AI assistantBetaModel providersCreate and update promptsCreate a promptManage promptsManage prompts programmaticallyPrompt template formatConfigur

Source: https://docs.langchain.com/langsmith/manage-prompts-programmatically


### Load data --> Docs --> Divide our text into chunks --> text --> convert chunks into vectors --> vectors(by using vector embedding) --> Store it in Vector Store DB

In [31]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

rct_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)

documents = rct_splitter.split_documents(docs)
documents

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/manage-prompts-programmatically', 'title': 'Manage prompts programmatically - Docs by LangChain', 'language': 'en'}, page_content='Manage prompts programmatically - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationCreate and update promptsManage prompts programmaticallyGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewQuickstartConceptsPolly AI assistantBetaModel providersCreate and update promptsCreate a promptManage promptsManage prompts programmaticallyPrompt template formatConfigure prompt settingsUse tools in a promptInclude multimodal content in a promptWrite your prompt with AIConnect to modelsTutorialsOptimize a classifierSync prompts with GitHubTest multi-turn conversationsOn this pageInstall packagesConfigure environment variablesPush a promptPull a promptPrompt cachingD

In [32]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9101.72it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [33]:
from langchain_community.vectorstores import FAISS

vector_store_db = FAISS.from_documents(documents,embeddings)


In [34]:
vector_store_db

In [55]:
### Querying

query = "Similar to pushing a prompt, you can also pull a prompt as a RunnableSequence"

result = vector_store_db.similarity_search(query)
print(result)
print(result[0])

[Document(id='dc1d9331-7bf7-4651-a520-e494f811f8df', metadata={'source': 'https://docs.langchain.com/langsmith/manage-prompts-programmatically', 'title': 'Manage prompts programmatically - Docs by LangChain', 'language': 'en'}, page_content='Manage prompts programmatically - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationCreate and update promptsManage prompts programmaticallyGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewQuickstartConceptsPolly AI assistantBetaModel providersCreate and update promptsCreate a promptManage promptsManage prompts programmaticallyPrompt template formatConfigure prompt settingsUse tools in a promptInclude multimodal content in a promptWrite your prompt with AIConnect to modelsTutorialsOptimize a classifierSync prompts with GitHubTest multi-turn conversationsOn this pageInstall packagesConfigure environment variabl

### Retrieval Chain, Document Chain

In [52]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.1-8b-instant")
print(llm)

profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True} client=<groq.resources.chat.completions.Completions object at 0x0000021D62D134D0> async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000021D62D13ED0> model_name='llama-3.1-8b-instant' model_kwargs={} groq_api_key=SecretStr('**********')


In [53]:
## Very Important Step in our RAG Applications

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """
    Answer the following question based only on the provided context:
    <context>
    {context}
    </context>
 
   """
)

document_chain = create_stuff_documents_chain(llm,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n    Answer the following question based only on the provided context:\n    <context>\n    {context}\n    </context>\n\n   '), additional_kwargs={})])
| ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000021D62D134D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000021D62D13ED0>, 

In [56]:
from langchain_core.documents import Document

document_chain.invoke({
    "input": "Similar to pushing a prompt, you can also pull a prompt as a RunnableSequence",
    "context": [Document(page_content="Similar to pushing a prompt, you can also pull a prompt as a RunnableSequence")]
})

'It seems like the provided context is about a programming or scripting interface that allows users to interact with a sequence of operations in a certain order or flow.\n\nBased on this context, it appears that a "pull a prompt" operation can be used to retrieve or initiate a sequence of operations, similar to how "pushing a prompt" would start a sequence. However, without more specific information, the exact nature and purpose of "pulling a prompt" as a RunnableSequence are unclear.'

### However we want the documents to first come from the retriever we just set up. That way, we can use the retriever to dynamiclly select the most relevant documents and pass those in for a given question

In [57]:
### Input --> Retriever( We can consider this as an interface) --> vectorStore DB


retreiver = vector_store_db.as_retriever()

from langchain_classic.chains import create_retrieval_chain

retrieval_chain = create_retrieval_chain(retreiver,document_chain)
retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000021D2F257CE0>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n    Answer the following question based only on the provided context:\n    <context>\n    {context}\n    </context>\n\n   '), additional_kwarg

In [59]:
### Get the response from the LLM

response = retrieval_chain.invoke({"input":"Similar to pushing a prompt, you can also pull a prompt as a RunnableSequence"})
print(response)
print(response['answer'])

{'input': 'Similar to pushing a prompt, you can also pull a prompt as a RunnableSequence', 'context': [Document(id='dc1d9331-7bf7-4651-a520-e494f811f8df', metadata={'source': 'https://docs.langchain.com/langsmith/manage-prompts-programmatically', 'title': 'Manage prompts programmatically - Docs by LangChain', 'language': 'en'}, page_content='Manage prompts programmatically - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationCreate and update promptsManage prompts programmaticallyGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewQuickstartConceptsPolly AI assistantBetaModel providersCreate and update promptsCreate a promptManage promptsManage prompts programmaticallyPrompt template formatConfigure prompt settingsUse tools in a promptInclude multimodal content in a promptWrite your prompt with AIConnect to modelsTutorialsOptimize a classifierSync pro

In [60]:
print(response['context'])

[Document(id='dc1d9331-7bf7-4651-a520-e494f811f8df', metadata={'source': 'https://docs.langchain.com/langsmith/manage-prompts-programmatically', 'title': 'Manage prompts programmatically - Docs by LangChain', 'language': 'en'}, page_content='Manage prompts programmatically - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationCreate and update promptsManage prompts programmaticallyGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewQuickstartConceptsPolly AI assistantBetaModel providersCreate and update promptsCreate a promptManage promptsManage prompts programmaticallyPrompt template formatConfigure prompt settingsUse tools in a promptInclude multimodal content in a promptWrite your prompt with AIConnect to modelsTutorialsOptimize a classifierSync prompts with GitHubTest multi-turn conversationsOn this pageInstall packagesConfigure environment variabl

In [37]:
result_and_score = vector_store_db.similarity_search_with_score(query)
result_and_score

[(Document(id='02f50e58-8685-4053-be63-545b5dceb951', metadata={'source': 'https://docs.langchain.com/langsmith/manage-prompts-programmatically', 'title': 'Manage prompts programmatically - Docs by LangChain', 'language': 'en'}, page_content='\u200bConfigure environment variables\nIf you already have LANGSMITH_API_KEY set to your current workspace’s api key from LangSmith, you can skip this step.\nOtherwise, get an API key for your workspace by navigating to Settings > API Keys > Create API Key in LangSmith.\nSet your environment variable.\nCopyexport LANGSMITH_API_KEY="lsv2_..."\n\nWhat we refer to as “prompts” used to be called “repos”, so any references to “repo” in the code are referring to a prompt.\n\u200bPush a prompt\nTo create a new prompt or update an existing prompt, you can use the push prompt method.\nPythonLangChain (Python)TypeScriptJavaCopyfrom langsmith import Client\nfrom langchain_core.prompts import ChatPromptTemplate\n\nclient = Client()\nprompt = ChatPromptTemplat